In [65]:
import os
import json
import numpy as np
import tensorflow as tf
from PIL import Image

MODEL_PATH = "ver2_tf_cbir_256\\ver2_model_float32.tflite"
DATASET_DIR = "DATA_NEW_merged"
OUTPUT_JSON = "ver2_32_embedding_dataset.json"

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input details: {input_details}")
print(f"Output details: {output_details}")

input_shape = input_details[0]['shape']
print(f"Expected input shape: {input_shape}")

Input details: [{'name': 'input', 'index': 0, 'shape': array([  1, 224, 224,   3]), 'shape_signature': array([  1, 224, 224,   3]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'Identity', 'index': 225, 'shape': array([  1, 256]), 'shape_signature': array([  1, 256]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Expected input shape: [  1 224 224   3]


In [66]:
import numpy as np

in_shape = input_details[0]["shape"]   # [1,224,224,3]
in_dtype = input_details[0]["dtype"]   # float32

dummy = np.zeros(in_shape, dtype=in_dtype)

interpreter.set_tensor(input_details[0]["index"], dummy)
interpreter.invoke()
print("OK, output shape:", interpreter.get_tensor(output_details[0]["index"]).shape)

OK, output shape: (1, 256)


In [67]:
from torchvision import transforms

val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
    
def preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = val_transform(img)
    img = np.array(img, dtype=np.float32) 
    img = np.transpose(img, (1, 2, 0))
    img = np.expand_dims(img, axis=0)
    return img.astype(np.float32)

def extract_embedding(image_path):
    img = preprocess_image(image_path)
    print(f"Preprocessed image shape: {img.shape}, dtype: {img.dtype}")
    if img is None:
        print(f"Failed to preprocess image: {image_path}")
        return None
    if list(img.shape) != list(input_shape):
        print(f"Unexpected input shape for image {image_path}: {img.shape}, expected: {input_shape}")
        return None
    
    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()
    embedding = interpreter.get_tensor(output_details[0]['index'])
    embedding = np.array(embedding).reshape(-1)
    return embedding.tolist()

In [68]:
valid_text = (".jpg", ".jpeg")

files = [f for f in os.listdir(DATASET_DIR) if os.path.isfile(os.path.join(DATASET_DIR, f)) and f.lower().endswith(valid_text)]
print(f"Found {len(files)} valid image files in {DATASET_DIR}")

result = []
error_files = 0

for idx, filename in enumerate(files):
    image_path = os.path.join(DATASET_DIR, filename)
    vec = extract_embedding(image_path)
    if vec is None:
        error_files += 1 
        continue 
    
    relative_id = f"..\\..\\DATA_NEW_merged\\{filename}"    
    
    result.append({
        "id": relative_id,
        "vector": vec
    })
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(files)} images...")
        
print(f"Finished processing. Total valid images: {len(result)}, Errors: {error_files}")
with open(OUTPUT_JSON, 'w') as f:
    json.dump(result, f, indent=2)
    
print(f'Done')

Found 3931 valid image files in DATA_NEW_merged
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preprocessed image shape: (1, 224, 224, 3), dtype: float32
Preproce

In [15]:
print(input_details[0]["shape"], input_details[0]["dtype"])


[  1 224 224   3] <class 'numpy.float32'>


In [47]:
import numpy as np

in_shape = input_details[0]["shape"]   # [1,224,224,3]
in_dtype = input_details[0]["dtype"]   # float32

dummy = np.zeros(in_shape, dtype=in_dtype)

interpreter.set_tensor(input_details[0]["index"], dummy)
interpreter.invoke()
print("OK, output shape:", interpreter.get_tensor(output_details[0]["index"]).shape)

OK, output shape: (1, 256)


In [1]:
import os

dataset_path = r"DATA_NEW_merged"
output_file = "DATA_NEW_imageMap.ts"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("export const imageMap: Record<string, any> = {\n")
    for file in os.listdir(dataset_path):
        if file.endswith(".jpg") or file.endswith(".jpeg") or file.endswith(".png"):
            f.write(f'  "{file}": require("./demo_dataset/DATA_NEW_merged/{file}"),\n')
    f.write("};\n")

print(f"✅ Đã tạo file map: {output_file}")

✅ Đã tạo file map: DATA_NEW_imageMap.ts
